# ⚽ Predicción del Campeón — Copa Mundial FIFA 2026
## Camino metodológico completo (v4 · sin fuga temporal)

Modelo de **clasificación multinomial** (Victoria / Empate / Derrota) que predice el resultado de un
partido, validado rigurosamente y usado para **simular** el Mundial (Monte Carlo).

**Lo distintivo:** (1) construimos ~12 variables candidatas y **dejamos que los datos elijan** el set
final (VIF + selección forward + significancia); (2) **evitamos la fuga temporal**: cada partido usa la
información disponible **a su fecha** (ranking FIFA y plantel de la versión FIFA vigente, no la de 2026).

**Decisiones de diseño:** datos 2022 → jun-2026, **sin localía** (variables simétricas equipo − rival),
**3 clases** (los empates se modelan).

**Índice:** Carga → EDA → **Selección de variables (sin fuga)** → Entrenamiento → Significancia → Matriz
de confusión → Tratamiento del empate → ROC → Modelos alternativos → Simulación → Poisson → Conclusiones.

## 0 · Configuración

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from collections import defaultdict, deque
sns.set_theme(style="whitegrid"); np.random.seed(42)
print("Listo.")

## 1 · Carga de datos y variables candidatas

Universo: **927 partidos de selecciones (2022 → 9-jun-2026)** (ESPN + API-Football, incluye amistosos
2026). Construimos **12 variables candidatas** (`diff` = equipo − rival, simétricas):

| Grupo | Variables | Fuente |
|---|---|---|
| **Fuerza / plantel (point-in-time)** | `fifa_diff`, `valor_diff`, `ea_diff`, `prof_diff`, `ligas_top_diff` | ranking FIFA por fecha + jugadores de la **versión FIFA vigente a la fecha** (15-23 histórico + FC26) |
| **Estilo de gol** | `concen_diff` | concentración del goleador (depende de pocos) |
| **Forma (walk-forward)** | `ppg_form_diff`, `racha_diff`, `form_cor/pos/sho_diff` | puntos/partido (últimos 10), racha, córners/posesión/tiros |
| **Head-to-head** | `h2h_diff` | historia entre los dos rivales (dif. de gol promedio) |

> **Clave anti-fuga:** las variables de plantel (`valor`, `prof`, `ligas_top`, `ea`) usan la versión
> FIFA **vigente al día del partido** (un partido de 2022 usa FIFA 22/23, no el plantel de 2026). Esto
> evita el *look-ahead bias*. `ppg_form`, `racha`, `h2h` y la forma detallada son **walk-forward**
> (solo partidos previos). `fifa` es por fecha (`merge_asof`).

In [ ]:
import build_features_full as BF
BF.main()   # genera data/modelado_full.csv (point-in-time + H2H, reproducible)
data_full = pd.read_csv("data/modelado_full.csv")
CAND = ["fifa_diff","valor_diff","ea_diff","prof_diff","ligas_top_diff","concen_diff",
        "ppg_form_diff","racha_diff","h2h_diff","form_cor_diff","form_pos_diff","form_sho_diff"]
data = data_full.dropna(subset=CAND).reset_index(drop=True)
print(f"\nUsables: {len(data)} | target:", data.resultado.value_counts(normalize=True).round(3).sort_index().to_dict())
data[["fifa_diff","prof_diff","ppg_form_diff","h2h_diff","resultado"]].head()

## 2 · Análisis exploratorio (EDA)

In [ ]:
cy = data[CAND].corrwith(data["resultado"]).sort_values(key=abs)
fig,ax=plt.subplots(figsize=(9,6)); sns.barplot(x=cy.values,y=cy.index,palette="vlag",ax=ax); ax.axvline(0,color="grey",lw=1)
ax.set_title("Correlación de cada variable candidata con el resultado"); plt.tight_layout(); plt.show()
print("Mayor correlación: ea, fifa, prof, ligas_top. La forma (ppg) y el h2h también aportan.")

## 3 · Selección de variables (sin fuga) — el corazón del trabajo

Tres pasos, dejando que **los datos** decidan: **(A)** quitar redundantes por **VIF**, **(B)** quedarnos
con las que **mejoran la predicción** (forward por CV log-loss), **(C)** confirmar **significancia**.

> ⚠️ **Por qué importa la fuga:** si usáramos el plantel **actual (2026)** para partidos de 2022-2025,
> el modelo "vería el futuro" (look-ahead bias) e inflaría artificialmente esas variables. Por eso
> usamos datos **point-in-time**. Como veremos, esto cambia el resultado: con datos limpios, la **forma
> reciente** entra al modelo (con el plantel actual quedaba tapada).

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold
cv=StratifiedKFold(5,shuffle=True,random_state=42); y=data["resultado"]
def vifs(fs): X=sm.add_constant(data[fs]); return {f:variance_inflation_factor(X.values,i+1) for i,f in enumerate(fs)}
def ll(fs): return -cross_val_score(Pipeline([("sc",StandardScaler()),("m",LogisticRegression(max_iter=2000))]),data[fs],y,cv=cv,scoring="neg_log_loss").mean()
# (A) VIF iterativo
feats=CAND[:]; print("(A) VIF iterativo (quito la peor hasta VIF<5):")
while True:
    v=vifs(feats); p=max(v,key=v.get)
    if v[p]<5: break
    print(f"   quito {p} (VIF={v[p]:.1f})"); feats.remove(p)
# (B) forward
print("(B) Selección forward (CV log-loss):")
sel=[]; rem=feats[:]; best=99
while rem:
    sc={f:ll(sel+[f]) for f in rem}; bf=min(sc,key=sc.get)
    if sc[bf]<best-0.001: sel.append(bf); rem.remove(bf); print(f"   + {bf:16} log-loss={sc[bf]:.4f}"); best=sc[bf]
    else: break
FINAL=sel
print(f"\n>>> SET FINAL (sin fuga): {FINAL}")
print(f"   log-loss con las 12 candidatas: {ll(CAND):.4f}  |  con el set final: {ll(FINAL):.4f}")
print(f"   VIF del set final:", {k:round(v,1) for k,v in vifs(FINAL).items()})

In [ ]:
# RENDICIÓN DE CUENTAS: probamos agregar CADA candidata al set final
base=ll(FINAL); Xv=sm.add_constant(data[CAND]); VIF={f:variance_inflation_factor(Xv.values,i+1) for i,f in enumerate(CAND)}
corr=data[CAND].corrwith(y); filas=[]
for f in CAND:
    if f in FINAL: filas.append([f,round(corr[f],2),round(VIF[f],1),"—","✅ seleccionada"]); continue
    gain=base-ll(FINAL+[f]); razon="VIF alto (colinealidad)" if VIF[f]>5 else ("no mejora / mete ruido" if gain<=0.001 else "aporte mínimo")
    filas.append([f,round(corr[f],2),round(VIF[f],1),round(gain,4),"❌ "+razon])
display(pd.DataFrame(filas,columns=["variable","corr_target","VIF","mejora_log_loss","destino"]))

**Resultado:** con datos **sin fuga**, los datos eligen **3 variables**: `fifa_diff` (ranking),
`prof_diff` (profundidad del plantel, *point-in-time*) y **`ppg_form_diff` (forma reciente)**.

**Hallazgos clave:**
- 🔑 **La forma reciente ahora SÍ entra al modelo y es significativa.** Cuando usábamos el plantel actual
  (con fuga), la forma quedaba tapada; al limpiar la fuga, su señal real aparece.
- `ea_diff` se descarta por **multicolinealidad** (VIF >10, colineal con FIFA).
- `valor`, `ligas_top`, `concen`, `h2h` se probaron pero **no mejoran** sobre las 3 elegidas (su señal ya
  está capturada). El `h2h` tiene correlación decente (0.34) pero es redundante con la fuerza relativa.

## 4 · División train/test y entrenamiento

In [ ]:
from sklearn.model_selection import train_test_split, cross_validate
data = data_full.dropna(subset=FINAL).reset_index(drop=True)
X, y = data[FINAL], data["resultado"]
Xtr,Xte,ytr,yte = train_test_split(X,y,test_size=0.25,random_state=42,stratify=y)
pipe = Pipeline([("sc",StandardScaler()),("lr",LogisticRegression(max_iter=2000))]).fit(Xtr,ytr)
print(f"Filas: {len(data)} | Train {len(Xtr)} / Test {len(Xte)} | variables: {FINAL}")

## 5 · Significancia (pseudo-R², p-values, coeficientes)

In [ ]:
mn = sm.MNLogit(ytr.values, sm.add_constant(StandardScaler().fit_transform(Xtr))).fit(disp=0)
print(f"Pseudo-R² (McFadden) = {mn.prsquared:.3f}\n")
Pp,Pv=np.asarray(mn.params),np.asarray(mn.pvalues); nombres=["const"]+FINAL
for k,clase in [(0,"EMPATE vs Derrota"),(1,"VICTORIA vs Derrota")]:
    t=pd.DataFrame({"variable":nombres,"coef":Pp[:,k],"p_value":Pv[:,k]}); t["signif"]=np.where(t.p_value<0.05,"✓","")
    print(f"--- {clase} ---"); print(t.round(3).to_string(index=False)); print()

**Las 3 variables son significativas** (p<0.05) para la Victoria, con signos coherentes: más ranking
FIFA, plantel más profundo y mejor forma reciente → más probabilidad de ganar.

## 6 · Matriz de confusión (test 25%)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, log_loss
pred=pipe.predict(Xte); proba=pipe.predict_proba(Xte)
cm=confusion_matrix(yte,pred,labels=[0,1,2])
fig,ax=plt.subplots(figsize=(5.5,4.5)); sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",cbar=False,
    xticklabels=["Derrota","Empate","Victoria"],yticklabels=["Derrota","Empate","Victoria"])
ax.set_xlabel("Predicho"); ax.set_ylabel("Real"); ax.set_title("Matriz de confusión (3 clases)"); plt.show()
print(classification_report(yte,pred,labels=[0,1,2],target_names=["Derrota","Empate","Victoria"]))
print(f"Accuracy={accuracy_score(yte,pred):.3f} | LogLoss={log_loss(yte,proba):.3f}")

El modelo acierta bien Victorias y Derrotas pero **casi nunca elige Empate por argmax** — la
siguiente sección explica por qué no es un error y cómo lo tratan los profesionales.

## 6b · Tratamiento del empate (cómo lo hacen los profesionales)

In [ ]:
def rps(P,y):
    Pc=np.cumsum(P,1); Oc=np.cumsum(np.eye(3)[np.asarray(y)],1); return np.mean(np.sum((Pc-Oc)**2,1))/(3-1)
print(f"1) Calibración: P(empate) media={proba[:,1].mean():.3f} vs tasa real={(yte==1).mean():.3f} -> casi perfecta")
print(f"2) RPS (métrica profesional V/E/D) = {rps(proba,yte.values):.3f}  (menor=mejor)")
for w in [None,"balanced"]:
    mm=Pipeline([("sc",StandardScaler()),("m",LogisticRegression(max_iter=2000,class_weight=w))]).fit(Xtr,ytr)
    print(f"3) class_weight={str(w):9}: {(mm.predict(Xte)==1).sum():2d} empates | LogLoss={log_loss(yte,mm.predict_proba(Xte)):.3f}")
print("   -> forzar empates empeora el log-loss. El modelo ya da bien la PROBABILIDAD del empate.")

Se evalúa con **RPS/log-loss** (no accuracy) y se usan las **probabilidades**. El estándar de las
casas de apuestas modela los goles (Poisson, sección 10).

## 7 · ROC y calibración (clase Victoria)

In [ ]:
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.calibration import calibration_curve
yv=(yte==2).astype(int); pv=proba[:,2]
fig,ax=plt.subplots(1,2,figsize=(13,5))
fpr,tpr,_=roc_curve(yv,pv); ax[0].plot(fpr,tpr,lw=2,label=f"AUC={auc(fpr,tpr):.3f}"); ax[0].plot([0,1],[0,1],"--",color="grey")
ax[0].set_title("ROC — Victoria (OVR)"); ax[0].set_xlabel("FPR"); ax[0].set_ylabel("TPR"); ax[0].legend()
po,pp=calibration_curve(yv,pv,n_bins=8); ax[1].plot(pp,po,"o-"); ax[1].plot([0,1],[0,1],"--",color="grey")
ax[1].set_title("Calibración — P(Victoria)"); ax[1].set_xlabel("Predicho"); ax[1].set_ylabel("Observado"); plt.tight_layout(); plt.show()
print(f"AUC (OVR macro) = {roc_auc_score(yte,proba,multi_class='ovr',average='macro'):.3f}")

## 8 · Modelos alternativos (CV k=5)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import PolynomialFeatures
mods={"Logística":Pipeline([("sc",StandardScaler()),("m",LogisticRegression(max_iter=2000))]),
 "Lasso (L1)":Pipeline([("sc",StandardScaler()),("m",LogisticRegression(penalty="l1",solver="saga",C=0.5,max_iter=4000))]),
 "Ridge (L2)":Pipeline([("sc",StandardScaler()),("m",LogisticRegression(penalty="l2",C=0.5,max_iter=2000))]),
 "Interacciones":Pipeline([("sc",StandardScaler()),("p",PolynomialFeatures(2,interaction_only=True,include_bias=False)),("m",LogisticRegression(max_iter=3000))]),
 "Random Forest":RandomForestClassifier(n_estimators=300,max_depth=6,random_state=42)}
res=[]
for nom,mod in mods.items():
    s=cross_validate(mod,X,y,cv=cv,scoring=["accuracy","neg_log_loss"])
    res.append({"Modelo":nom,"Accuracy":s["test_accuracy"].mean(),"LogLoss":-s["test_neg_log_loss"].mean()})
tabla=pd.DataFrame(res).set_index("Modelo").sort_values("LogLoss")
display(tabla.round(4).style.highlight_min(subset=["LogLoss"],color="lightgreen"))
best=tabla.index[0]; print(f">>> Mejor: {best}")

## 9 · Simulación del Mundial 2026 (Monte Carlo)

In [ ]:
import mundial2026 as M
final_model = mods[best].fit(X,y)
fifa = pd.read_csv("data/fifa_ranking_historico.csv",parse_dates=["date"])[["team","total_points","date"]]
cur=pd.read_csv("data/fifa_ranking_2026.csv"); fifa=pd.concat([fifa,pd.DataFrame({"team":cur.team,"total_points":cur.points,"date":pd.Timestamp("2026-06-08")})]).sort_values("date")
def fifa_pts(t):
    s=fifa[fifa.team==BF.BD.MAP_FIFA.get(t,t)]; return s.total_points.iloc[-1] if len(s) else np.nan
VERS,VF=BF.atributos_por_version(); PROF=VERS[VF[-1]]   # FC26 = plantel actual (point-in-time para 2026)
def prof(t): return PROF.get(BF.BD.MAP_EA.get(t,t),{}).get("prof",np.nan)
rr=pd.read_csv("data/results.csv",parse_dates=["date"]).sort_values("date"); rr=rr[rr.date>="2022-01-01"].dropna(subset=["home_score","away_score"])
ppgd=defaultdict(lambda:deque(maxlen=10))
for r in rr.itertuples():
    rh,ra=(3,0) if r.home_score>r.away_score else ((0,3) if r.home_score<r.away_score else (1,1))
    ppgd[r.home_team].append(rh); ppgd[r.away_team].append(ra)
def ppgf(t): return np.mean(ppgd[t]) if ppgd[t] else np.nan
teams48=[t for g in M.GRUPOS.values() for t in g]
FE={t:dict(fifa=fifa_pts(t),prof=prof(t),ppg=ppgf(t)) for t in teams48}
med={f:data[f].median() for f in FINAL}
def vec(i,j):
    a,b=FE[i],FE[j]
    v={"fifa_diff":a["fifa"]-b["fifa"],"prof_diff":a["prof"]-b["prof"],"ppg_form_diff":a["ppg"]-b["ppg"]}
    return {f:(v[f] if pd.notna(v[f]) else med[f]) for f in FINAL}
pares=[(i,j) for i in teams48 for j in teams48 if i!=j]
PP=final_model.predict_proba(pd.DataFrame([vec(i,j) for i,j in pares]))
PROB={(i,j):(PP[k][2],PP[k][1],PP[k][0]) for k,(i,j) in enumerate(pares)}
print(f"Matriz de {len(PROB)} pares lista. P(empate) media={np.mean([v[1] for v in PROB.values()]):.2f}")

In [ ]:
rng=np.random.default_rng(42)
def grupo(eqs):
    pts={t:0 for t in eqs}; gd={t:0 for t in eqs}
    for x in range(len(eqs)):
        for z in range(x+1,len(eqs)):
            i,j=eqs[x],eqs[z]; pi,pe,pj=PROB[(i,j)]; u=rng.random()
            if u<pi: pts[i]+=3;gd[i]+=1;gd[j]-=1
            elif u<pi+pe: pts[i]+=1;pts[j]+=1
            else: pts[j]+=3;gd[j]+=1;gd[i]-=1
    return sorted(eqs,key=lambda t:(pts[t],gd[t],rng.random()),reverse=True)
def elim(i,j):
    pi,pe,pj=PROB[(i,j)]; s=pi+pj; return i if rng.random()<pi/s else j
def sim():
    pr,se,te_={},{},[]
    for g,eqs in M.GRUPOS.items():
        o=grupo(eqs); pr[g],se[g]=o[0],o[1]; te_.append((g,o[2]))
    tg={g:t for g,t in sorted(te_,key=lambda x:rng.random())[:8]}; asg=M.asignar_terceros(set(tg))
    sl=lambda s,n: tg[asg[n]] if isinstance(s,tuple) else (pr[s[1]] if s[0]=="1" else se[s[1]])
    W={n:elim(sl(a,n),sl(b,n)) for n,(a,b) in M.R32.items()}
    for rd in (M.R16,M.QF,M.SF):
        for n,(p,q) in rd.items(): W[n]=elim(W[p],W[q])
    return elim(W[M.FINAL[0]],W[M.FINAL[1]])
from collections import Counter
N=20000; champ=Counter(sim() for _ in range(N))
final=pd.DataFrame([{"Selección":t,"P_campeon":champ[t]/N} for t in teams48]).sort_values("P_campeon",ascending=False).reset_index(drop=True)
print(f"{N:,} Mundiales simulados.")
display(final.head(12).style.format({"P_campeon":"{:.1%}"}))
top=final.head(12).iloc[::-1]; fig,ax=plt.subplots(figsize=(9,6)); ax.barh(top["Selección"],top["P_campeon"]*100,color="#0b3d91")
for k,v in enumerate(top["P_campeon"]*100): ax.text(v+0.1,k,f"{v:.1f}%",va="center")
ax.set_xlabel("Probabilidad de ser campeón (%)"); ax.set_title("Mundial 2026 — Predicción final (sin fuga)"); plt.tight_layout(); plt.show()

## 10 · Modelo Poisson de goles (enfoque casas de apuestas)

El estándar profesional modela los **goles** (Poisson) y deriva V/E/D de la grilla de marcadores;
los empates surgen naturalmente y permite predecir el **marcador exacto**.

In [ ]:
from scipy.stats import poisson
dtr,dte=train_test_split(data,test_size=0.25,random_state=42,stratify=data.resultado)
pois=sm.GLM(dtr["goles"],sm.add_constant(dtr[FINAL]),family=sm.families.Poisson()).fit()
b0=pois.params["const"]; bet=pois.params[FINAL].values
def lambdas(fr): s=float(np.dot(bet,fr)); return np.exp(b0+s),np.exp(b0-s)
def wdl(li,lj,m=8):
    pi=poisson.pmf(np.arange(m+1),li);pj=poisson.pmf(np.arange(m+1),lj);Mx=np.outer(pi,pj)
    return np.array([np.triu(Mx,1).sum(),np.trace(Mx),np.tril(Mx,-1).sum()])
pp=np.array([wdl(*lambdas(r[FINAL].values.astype(float))) for _,r in dte.iterrows()]); pp/=pp.sum(1,keepdims=True)
pl=pipe.predict_proba(dte[FINAL]); yv=dte.resultado.values
comp=pd.DataFrame({"Modelo":["Logística multinomial","Poisson de goles"],
  "Accuracy":[accuracy_score(yv,pl.argmax(1)),accuracy_score(yv,pp.argmax(1))],
  "LogLoss":[log_loss(yv,pl),log_loss(yv,pp)],"RPS":[rps(pl,yv),rps(pp,yv)]}).set_index("Modelo")
display(comp.round(4).style.highlight_min(subset=["LogLoss","RPS"],color="lightgreen"))
print("Marcadores más probables (Poisson):")
for i,j in [("Spain","Argentina"),("France","Brazil"),("England","Germany")]:
    li,lj=lambdas(np.array([vec(i,j)[f] for f in FINAL])); m=np.outer(poisson.pmf(np.arange(7),li),poisson.pmf(np.arange(7),lj))
    x,y_=np.unravel_index(m.argmax(),m.shape); print(f"  {i} {x}-{y_} {j} ({m[x,y_]*100:.0f}%)")

## 11 · Conclusiones y limitaciones

**Metodología:**
- **Sin fuga temporal:** plantel point-in-time (versión FIFA vigente a cada fecha) — un partido de 2022
  no usa el plantel de 2026. Esto cambió el resultado: la **forma reciente** entra al modelo.
- **Selección de variables data-driven:** de 12 candidatas, sobreviven **3** (FIFA + profundidad + forma)
  por VIF + forward + significancia. Probamos H2H, valor, ligas-top, concentración → no aportan extra.
- Clasificación **multinomial (V/E/D)** con el empate modelado (RPS como métrica).
- Validación **75/25** + **CV k=5**; comparación de 5 modelos + **Poisson de goles** (coinciden → robusto).

**Hallazgo central:** detectar y corregir la **fuga temporal** fue clave — con el plantel actual, la
forma reciente quedaba tapada; con datos point-in-time, su señal real aparece y es significativa.

**Limitaciones:** los datos históricos de plantel llegan a FIFA 23 (2024-25 usan la última versión previa,
sin fuga pero algo desactualizada); el xG cubre 39/48 (no entró por cobertura); no considera lesiones.